In [ ]:
!pip install transformers sentence-transformers


In [ ]:

!pip install  langchain langchain-community langchain-core langchain-groq google-search-results gradio python-dotenv openpyxl faiss-cpu pypdf pdfplumber gtts deep_translator pydub openai-whisper ffmpeg-python

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 794.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 1.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# !pip install -q transformers sentence-transformers
# !pip install -q langchain langchain-community langchain-core langchain-groq google-search-results gradio python-dotenv openpyxl faiss-cpu pypdf pdfplumber gtts deep_translator pydub openai-whisper ffmpeg-python

In [ ]:
!pip install langchain-docling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user
^C


In [ ]:
# ==============================================================================
# 1. DEPENDENCY INSTALLATION
# ==============================================================================
import sys
import subprocess

print("📦 Installing Docling, Gradio PDF Viewer, FAISS, and dependencies...")
packages = [
    "docling>=2.0.0",
    "gradio-pdf>=0.0.24",
    "google-search-results",
    "transformers",
    "sentence-transformers",
    "langchain",
    "langchain-community",
    "langchain-core",
    "langchain-groq",
    "gradio>=4.15.0",
    "python-dotenv",
    "openpyxl",
    "faiss-cpu",
    "pypdf",
    "pdfplumber",
    "gtts",
    "deep_translator",
    "pydub",
    "openai-whisper",
    "ffmpeg-python",
    "pandas"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✅ Libraries installed successfully!\n")
# ==============================================================================
# 2. SETUP MODELS, ENVIRONMENT & IMPORTS
# ==============================================================================
import os
import csv
import warnings
import pandas as pd

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import gradio as gr
from gradio_pdf import PDF
from deep_translator import GoogleTranslator
from gtts import gTTS
import whisper
from serpapi import GoogleSearch
import base64

# Docling Parser
# from docling.document_converter import DocumentConverter
# LangChain Imports
from langchain_community.document_loaders import PDFPlumberLoader, PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage
from langchain_groq import ChatGroq

from google.colab import userdata
from google.colab.userdata import TimeoutException

# Retrieve API Keys
groq_api_key = userdata.get('GROQ_API_KEY')

try:
    serpapi_key = userdata.get('SERPAPI_API_KEY')
except TimeoutException:
    print("Colab userdata.get for SERPAPI_API_KEY timed out. Attempting to retrieve from environment variables.")
    serpapi_key = os.getenv('SERPAPI_API_KEY')

stt_model = whisper.load_model("base")

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.3,
    groq_api_key=groq_api_key if groq_api_key else "DUMMY_KEY_FOR_INIT"
)

# docling_converter = DocumentConverter()



CSV_FILE = "candidate_data.csv"
if not os.path.exists(CSV_FILE):
    with open(CSV_FILE, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Name", "Email", "Phone", "Course Interest"])

SALES_PROMPTS = {
    "PragyanAI Student Counselor": """You are Aarav, an Academic & Career Advisor for PragyanAI.
Goal: Guide prospective students to enroll in the 18-Month AI/GenAI Program (6 Month Offline Training + 12 Month Placement Drive).

Strict Rule: Answer pricing, fee structures, curriculum details, and salary potential based on the Document Context and Search Results below.

Retrieved Context:
{context}

Behavior Guidelines:
1. Be encouraging, empathetic, and focus on practical "builder" skill transformation.
2. Highlight key advantages: 100+ projects, 48-hour hackathons, risk-shared pricing (pay-after-placement success fee), and direct mentorship under Sateesh Ambesange.""",

    "PragyanAI Institutional / CoE Advisor": """You are Dr. Kavita, Institutional Relations Lead at PragyanAI.
Goal: Partner with engineering colleges to solve the education trap and transform students into product builders.

Retrieved Context:
{context}""",

    "PragyanAI Enterprise AI & Placement Lead": """You are Rohan, Enterprise Placement & Venture Lead at PragyanAI.
Goal: Connect hiring partners and enterprise leaders with top-tier PragyanAI builders.

Retrieved Context:
{context}"""
}

LANGUAGES = {
    "English": "en",
    "Hindi": "hi",
    "Kannada": "kn",
    "Telugu": "te",
    "Tamil": "ta",
    "Spanish": "es",
    "French": "fr"
}

vector_store = None
raw_extracted_text = ""
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
uploaded_file_paths = {}  # Map: file_name -> path


def initialize_default_knowledge_base():
    global vector_store, raw_extracted_text

    # Auto-load default FAQ Excel if available locally
    if os.path.exists("pragyan_faq_prices.xlsx"):
        import pandas as pd
        excel_df = pd.read_excel("pragyan_faq_prices.xlsx")
        text_content = excel_df.to_string()
        raw_extracted_text = text_content

        from langchain_core.documents import Document
        doc = Document(page_content=text_content, metadata={"source": "pragyan_faq_prices.xlsx"})

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        chunks = text_splitter.split_documents([doc])
        vector_store = FAISS.from_documents(chunks, embeddings)
        print("✅ Pre-loaded default PragyanAI knowledge base.")

# Auto-run on startup
initialize_default_knowledge_base()
# ==============================================================================
# 3. SERPAPI LIVE WEB SEARCH MODULE
# ==============================================================================
def google_search_api(query, num_results=3):
    """Executes live web search using SerpAPI."""
    api_key = userdata.get('SERPAPI_API_KEY') or serpapi_key

    if not api_key:
        return "[SERPAPI_API_KEY not configured in environment variables]"

    try:
        params = {
            "engine": "google",
            "q": query,
            "num": num_results,
            "api_key": api_key,
        }

        search = GoogleSearch(params)
        results_dict = search.get_dict()

        organic_results = results_dict.get("organic_results", [])
        if not organic_results:
            return "No web results found."

        search_snippets = []
        for idx, item in enumerate(organic_results[:num_results], 1):
            title = item.get("title", "No Title")
            snippet = item.get("snippet", "No snippet available.")
            link = item.get("link", "")
            search_snippets.append(f"[{idx}] {title}\nSnippet: {snippet}\nURL: {link}")

        return "\n\n".join(search_snippets)

    except Exception as e:
        return f"[SerpAPI Search Error: {str(e)}]"
# ==============================================================================
# 4. MULTI-ENGINE PDF EXTRACTION & VECTOR STORE INDEXING
# ==============================================================================
def extract_text_multi_engine(file_path):
    """Attempts extraction using Docling OCR -> PyPDF -> PDFPlumber Fallback."""
    text_content = ""

    # Engine 1: IBM Docling (OCR & Layout Aware) - Uncomment if enabled
    # try:
    #     result = docling_converter.convert(file_path)
    #     text_content = result.document.export_to_markdown()
    #     if len(text_content.strip()) > 50:
    #         return text_content, "Docling OCR"
    # except Exception as e:
    #     print(f"Docling Engine skipped/failed: {e}")

    # Engine 2: PyPDF (Fast text extraction)
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        text_content = "\n\n".join([d.page_content for d in docs])
        if len(text_content.strip()) > 50:
            return text_content, "PyPDF"
    except Exception as e:
        print(f"PyPDF Engine skipped/failed: {e}")

    # Engine 3: PDFPlumber Fallback (Accurate text/layout extraction)
    try:
        loader = PDFPlumberLoader(file_path)
        docs = loader.load()
        text_content = "\n\n".join([d.page_content for d in docs])
        if len(text_content.strip()) > 50:
            return text_content, "PDFPlumber"
    except Exception as e:
        print(f"PDFPlumber Engine failed: {e}")

    return "", "Failed"


def load_documents_into_vectorstore(files):
    global vector_store, raw_extracted_text, uploaded_file_paths

    # Return exactly 3 items to match Gradio outputs: (status, dropdown_update, file_path)
    if not files:
        return "No files uploaded.", gr.update(choices=[], value=None), None

    documents = []
    text_buffer = []
    file_names = []
    uploaded_file_paths.clear()

    for file_obj in files:
        file_path = file_obj.name if hasattr(file_obj, "name") else file_obj
        file_name = os.path.basename(file_path)
        file_names.append(file_name)
        uploaded_file_paths[file_name] = file_path

        try:
            if file_path.endswith(".pdf"):
                text_content, engine_used = extract_text_multi_engine(file_path)
                if text_content:
                    documents.append(
                        Document(
                            page_content=text_content,
                            metadata={"source": file_name, "engine": engine_used}
                        )
                    )
                    text_buffer.append(
                        f"--- Document: {file_name} (Parsed via {engine_used})---\n{text_content}"
                    )

            elif file_path.endswith((".xlsx", ".xls")):
                df = pd.read_excel(file_path)
                text_content = df.to_string()
                documents.append(
                    Document(
                        page_content=text_content,
                        metadata={"source": file_name, "engine": "Pandas"}
                    )
                )
                text_buffer.append(text_content)

        except Exception as e:
            return (
                f"❌ Error loading file {file_name}: {str(e)}",
                gr.update(choices=[], value=None),
                None
            )

    if not documents:
        return (
            "⚠️ Could not extract text from uploaded documents.",
            gr.update(choices=[], value=None),
            None
        )

    raw_extracted_text = "\n\n".join(text_buffer)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = text_splitter.split_documents(documents)

    if vector_store is None:
        vector_store = FAISS.from_documents(chunks, embeddings)
    else:
        vector_store.add_documents(chunks)

    first_pdf_path = uploaded_file_paths.get(file_names[0], None)

    return (f"✅ Loaded {len(files)} file(s) ac")
# ==============================================================================
# 5. RAG & VOICE SYNTHESIS ENGINE
# ==============================================================================
def respond(query, persona="PragyanAI Student Counselor", google_search=True, web_search=False, history=None):
    global vector_store

    if not query or not query.strip():
        return "Please ask a question."

    pdf_context = ""
    web_context = ""

    if vector_store is not None:
        try:
            retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 12})
            relevant_docs = retriever.invoke(query)
            if relevant_docs:
                pdf_context = "\n---\n".join([doc.page_content for doc in relevant_docs])
        except Exception as e:
            print(f"Retrieval Error: {e}")

    if google_search or web_search:
        web_context = google_search_api(query, num_results=3)

    full_retrieved_context = ""
    if pdf_context:
        full_retrieved_context += f"--- PDF DOCUMENT CONTEXT ---\n{pdf_context}\n\n"
    if web_context:
        full_retrieved_context += f"--- GOOGLE WEB SEARCH RESULTS ---\n{web_context}"

    if not full_retrieved_context:
        full_retrieved_context = "No relevant context found in documents or web."

    prompt_template = SALES_PROMPTS.get(persona, SALES_PROMPTS["PragyanAI Student Counselor"])
    formatted_system_prompt = prompt_template.format(context=full_retrieved_context)

    try:
        messages = [
            SystemMessage(content=formatted_system_prompt),
            ("human", query),
        ]
        response = llm.invoke(messages)
        return response.content
    except Exception as e:
        return f"### 📑 Extracted Context:\n{full_retrieved_context}\n\n*(Error: {str(e)})*"

def process_voice_and_chat(user_text, voice_file, target_lang, persona, google_search, web_search):
    query = ""

    if voice_file is not None:
        transcription = stt_model.transcribe(voice_file, fp16=False)
        query = transcription.get("text", "").strip()

    if not query and user_text:
        query = user_text.strip()

    if not query:
        return "Please ask a question via voice or text input.", None

    lang_code = LANGUAGES.get(target_lang, "en")

    if lang_code != "en":
        try:
            processed_query = GoogleTranslator(source="auto", target="en").translate(query)
        except Exception:
            processed_query = query
    else:
        processed_query = query

    raw_response = respond(
        query=processed_query,
        persona=persona,
        google_search=google_search,
        web_search=web_search
    )

    if lang_code != "en":
        try:
            final_response = GoogleTranslator(source="auto", target=lang_code).translate(raw_response)
        except Exception:
            final_response = raw_response
    else:
        final_response = raw_response

    tts_audio_path = "response.mp3"
    try:
        tts = gTTS(text=final_response, lang=lang_code, slow=False)
        tts.save(tts_audio_path)
    except Exception:
        tts_audio_path = None

    return final_response, tts_audio_path

def save_candidate_data(name, email, phone, course):
    if not name or not email:
        return "⚠️ Please provide at least a Name and Email.", CSV_FILE
    with open(CSV_FILE, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([name, email, phone, course])
    return f"✅ Saved record for {name}!", CSV_FILE

def on_pdf_selected(paper_name):
    """Updates the PDF viewer when a file is selected from the dropdown."""
    if paper_name in uploaded_file_paths:
        return uploaded_file_paths[paper_name]
    return None

def chat_interface_respond(message, history, persona, google_search, web_search):
    return respond(message, persona, google_search, web_search, history)
!pip install PyMuPDF


import base64
import os
import fitz  # PyMuPDF
import gradio as gr
from PIL import Image


def get_pdf_page_count(pdf_path):
  """Gets total pages in a PDF file using PyMuPDF."""
  if not pdf_path or not os.path.exists(pdf_path):
    return 0
  try:
    doc = fitz.open(pdf_path)
    count = len(doc)
    doc.close()
    return count
  except Exception as e:
    print(f"Error reading PDF page count: {e}")
    return 0


def render_pdf_page_as_image(pdf_name, page_num):
  """Renders a specific PDF page as a PIL Image."""
  if not pdf_name or pdf_name not in uploaded_file_paths:
    return None, 1, 0, "**Page 0 of 0**"

  file_path = uploaded_file_paths[pdf_name]
  total_pages = get_pdf_page_count(file_path)

  if total_pages == 0:
    return None, 1, 0, "**Error loading PDF pages**"

  # Clamp page number within bounds
  page_num = max(1, min(page_num, total_pages))

  try:
    doc = fitz.open(file_path)
    # PyMuPDF page numbers are 0-indexed
    page = doc.load_page(page_num - 1)

    # Scale matrix for high quality rendering (2.0 = 2x zoom/clarity)
    pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()

    page_label = f"**Page {page_num} of {total_pages}**"
    return img, page_num, total_pages, page_label

  except Exception as e:
    print(f"Error rendering page to image: {e}")
    return None, page_num, total_pages, f"**Error loading page {page_num}**"


def navigate_pdf(pdf_name, current_page, action):
  """Handles Next, Previous, Start, End, and Direct Page Jumps."""
  if not pdf_name or pdf_name not in uploaded_file_paths:
    return None, 1, "**Page 0 of 0**"

  file_path = uploaded_file_paths[pdf_name]
  total_pages = get_pdf_page_count(file_path)

  if action == "start":
    target_page = 1
  elif action == "end":
    target_page = total_pages
  elif action == "next":
    target_page = current_page + 1
  elif action == "prev":
    target_page = current_page - 1
  else:
    target_page = current_page  # Direct jump

  target_page = max(1, min(target_page, total_pages))
  img, active_page, total, label = render_pdf_page_as_image(
      pdf_name, target_page
  )

  return img, active_page, label

In [ ]:
# ---------------------------------------------------------------------------
# 6. Gradio UI Blocks Layout
# ---------------------------------------------------------------------------
with gr.Blocks(
    title="PragyanAI Intelligent Assistant", theme=gr.themes.Soft()
) as demo:
  gr.Markdown("# 🎓 PragyanAI Conversational Sales & FAQ Assistant")
  gr.Markdown(
      "Answers program questions with **Voice Processing, Google API Search,"
      " and RAG Extraction**."
  )

  with gr.Tabs():
    # TAB 1: Voice & Multilingual Assistant
    with gr.Tab("🎙️ Voice & Document Assistant"):
      with gr.Row():
        with gr.Column(scale=1):
          persona_selector = gr.Dropdown(
              choices=list(SALES_PROMPTS.keys()),
              value="PragyanAI Student Counselor",
              label="Select Persona",
              interactive=True,
          )

          language_selector = gr.Dropdown(
              choices=list(LANGUAGES.keys()),
              value="English",
              label="Output Language (Translation & TTS)",
              interactive=True,
          )

          enable_google = gr.Checkbox(
              label="Enable Live Google API Search", value=True
          )
          enable_web_search = gr.Checkbox(
              label="Enable Deep Search", value=False
          )

          file_uploader = gr.File(
              label="Upload Program PDF or Excel Sheets",
              file_count="multiple",
              file_types=[".pdf", ".xlsx", ".xls"],
          )
          upload_status = gr.Textbox(
              label="Vector Store Status",
              value="No documents uploaded yet.",
              interactive=False,
          )
          file_uploader.change(
              fn=load_documents_into_vectorstore,
              inputs=[file_uploader],
              outputs=[upload_status],
          )

        with gr.Column(scale=2):
          voice_input = gr.Audio(
              label="Speak Your Question 🎙️", type="filepath"
          )
          user_input_text = gr.Textbox(
              label="Or Type Your Question 💬",
              placeholder="Ask about fees, curriculum, eligibility...",
          )

          submit_btn = gr.Button("Submit Query", variant="primary")

          text_output = gr.Textbox(
              label="Assistant Response", lines=10, interactive=False
          )
          voice_output = gr.Audio(label="Voice Response 🔊", autoplay=True)

          submit_btn.click(
              fn=process_voice_and_chat,
              inputs=[
                  user_input_text,
                  voice_input,
                  language_selector,
                  persona_selector,
                  enable_google,
                  enable_web_search,
              ],
              outputs=[text_output, voice_output],
          )

    # TAB 2: Extracted Program Summary Hub
    with gr.Tab("📋 Extracted Program Summary"):
      gr.Markdown("### 📊 Direct Extracted Information Hub")
      gr.Markdown(
          "Click a button below to view targeted details parsed directly from"
          " your uploaded document."
      )

      with gr.Row():
        btn_curriculum = gr.Button(
            "📚 Curriculum & Syllabus", variant="secondary"
        )
        btn_highlights = gr.Button("⭐ Program Highlights", variant="secondary")
        btn_price = gr.Button("💰 Price & Fees", variant="secondary")
        btn_why = gr.Button("🎯 Why Program", variant="secondary")
        btn_other = gr.Button("📌 Other Key Points", variant="secondary")

      summary_display = gr.Markdown(
          "Upload a PDF/Excel file in Tab 1, then click any category above."
      )

      btn_curriculum.click(
          fn=lambda: extract_section_info("Curriculum"),
          outputs=[summary_display],
      )
      btn_highlights.click(
          fn=lambda: extract_section_info("Program Highlights"),
          outputs=[summary_display],
      )
      btn_price.click(
          fn=lambda: extract_section_info("Price & Fees"),
          outputs=[summary_display],
      )
      btn_why.click(
          fn=lambda: extract_section_info("Why Program"),
          outputs=[summary_display],
      )
      btn_other.click(
          fn=lambda: extract_section_info("Other Key Points"),
          outputs=[summary_display],
      )

    # TAB 3: Interactive Text Chat
    with gr.Tab("💬 Interactive Text Chat"):
      chatbot_ui = gr.ChatInterface(
          fn=chat_interface_respond,
          additional_inputs=[persona_selector, enable_google, enable_web_search],
      )

    # TAB 4: College Info & Lead Capture
    with gr.Tab("🏛️ College Info & Lead Capture"):
      with gr.Accordion("📌 PragyanAI College & Program Information", open=True):
        gr.Markdown("""
                * **Institution:** PragyanAI Smart Technology
                * **Focus Areas:** Deep-Tech, Multi-Agent AI Systems, EDA Lifecycle Automation
                * **Key Program:** 18-Month AI/GenAI Program (6M Offline + 12M Placement)
                * **Contact:** admissions@pragyan.ai
                """)

      gr.Markdown("---")
      gr.Markdown("### 📝 Candidate Registration & Record Collection")
      with gr.Row():
        cand_name = gr.Textbox(
            label="Full Name", placeholder="e.g. Rahul Verma"
        )
        cand_email = gr.Textbox(
            label="Email Address", placeholder="e.g. rahul@example.com"
        )
        cand_phone = gr.Textbox(
            label="Phone Number", placeholder="e.g. +91 9876543210"
        )
        cand_course = gr.Dropdown(
            choices=[
                "18-Month AI/GenAI Program",
                "Executive AI Leadership",
                "GenAI Certification",
                "Other",
            ],
            label="Course Interest",
            value="18-Month AI/GenAI Program",
        )

      save_btn = gr.Button("Save Candidate Information", variant="primary")
      status_output = gr.Textbox(label="Status", interactive=False)
      csv_download = gr.File(label="Download CSV Records", value=CSV_FILE)

      save_btn.click(
          fn=save_candidate_data,
          inputs=[cand_name, cand_email, cand_phone, cand_course],
          outputs=[status_output, csv_download],
      )

if __name__ == "__main__":
  demo.launch(share=True, debug=True)

NameError: name 'gr' is not defined